In [ ]:
%%javascript
IPython.OutputArea.auto_scroll_threshold = 9999;

In [ ]:
#Imports

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
%matplotlib inline



In [ ]:
# path defines
gtpath = "./groundtruth"
respath = "./results_individual"
processed_resdir="./results_processed"
plot_dir="./res_plots"

for p in [processed_resdir, plot_dir]:
    if not os.path.exists(p):
        os.mkdir(p)


# DEFINE MODEL NAME + PROMPT LANGUAGE HERE: gpt3_de_, gpt4_de_, gpt3_en_, gpt4_en_, leo_de_
model = "gpt3_en_"



In [ ]:
# Naming conventions/defines
tasks_subj_en = ["Attractiveness", "Likeability", "Morality", "Intelligence", "Work Ethic","Arrogance","Laziness","Ungratefulness","Narrowmindedness","Insecurity","Xenophobia","Materialism","Ignorance"]
tasks_obj_en = ["Precipitation", "Average Temperature", "Population Density", "Infant Mortality Rate", "Nighttime Light Intensity", "Average Age", "Education"]
tasks_objrel_en = [t+" (Rating)" for t in tasks_obj_en]+["Body Temperature (Rating)"]
tasks_obj_en_neg=['Precipitation', 'Infant Mortality Rate', 'Nighttime Light Intensity', 'Average Age']
cmap_abs = ["Purples", "Greens", "Oranges"]
cmap_rel=['coolwarm','coolwarm','coolwarm']

east_states=['Sachsen (Saxony)', 'Thüringen (Thuringia)', 'Sachsen-Anhalt (Saxony-Anhalt)', 'Brandenburg', 'Mecklenburg-Vorpommern']

region_dict = {'Bayern':'Bayern (Bavaria)',
               'Niedersachsen':'Niedersachsen (Lower Saxony)',
               'Rheinland-Pfalz':'Rheinland-Pfalz (Rhineland-Palatinate)',
               'Sachsen':'Sachsen (Saxony)',
               'Thüringen':'Thüringen (Thuringia)',
               'Nordrhein-Westfalen':'Nordrhein-Westfalen (North Rhine-Westphalia)',
               'Sachsen-Anhalt':'Sachsen-Anhalt (Saxony-Anhalt)'
               }

In [ ]:
# Plotting function on Germany map
def plot_on_ger(df, col, title, saveto, adapt_names = True, cmap="Blues", vmin=None, vmax=None):
    map_dataframe = gpd.read_file('DEU_adm/DEU_adm1.shx')
    if adapt_names:
        map_dataframe = map_dataframe.replace({"NAME_1": region_dict})
    
    merged_dataframe = map_dataframe.merge(df, left_on=["NAME_1"], right_on=df.index)
 
    fig, ax = plt.subplots(1, 1, figsize=(6.42, 8.33))
    plot = merged_dataframe.plot(column=col, cmap=cmap, edgecolor='black', linewidth=0.5, ax=ax, vmin=vmin, vmax=vmax)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm._A = [] 
    #cbar = fig.colorbar(sm, ax=ax, location="bottom")
    plt.tight_layout(pad=0.0)
    plt.margins(0,0)
    plt.savefig(saveto, dpi=300)
    # uncomment this and comment plt.close to see plots in notebook
    #plt.show()
    plt.close(fig)

In [ ]:



# Initialize a dictionary to hold the DataFrame for each column
column_dataframes = {}

# Load each file and append its columns to the corresponding DataFrame
for run in range(1,6):
    filename = respath+'/'+model+str(run)+'.csv'
    print(filename)
    # Load the current iteration's data
    current_df = pd.read_csv(filename)
    current_df.set_index("Region", inplace=True)
    
    # For each column in this DataFrame, append the data to the appropriate DataFrame in the dictionary
    for column in current_df.columns:
        # If the column DataFrame does not exist, create it
        if column not in column_dataframes:
            column_dataframes[column] = pd.DataFrame(index=current_df.index)
        
        # Append the current column data as a new column in the corresponding DataFrame
        # The new column's name will be the iteration number, extracted from the filename
        iteration_name = model+str(run)
        column_dataframes[column][iteration_name] = current_df[column]

# Now column_dataframes contains a DataFrame for each column across all iterations
# Each DataFrame's columns are the iterations, and rows are the data entries
# "Absolute" Education value by models when prompted to be between 0 and 100 is generally 10x that of the rating
column_dataframes["Education"] = column_dataframes["Education (Rating)"].mul(10.)


In [ ]:
#load ground truth values

groundtruths = pd.DataFrame()

for obj_task in tasks_obj_en:
    # Load the current iteration's data
    current_df = pd.read_csv(gtpath+'/'+"GT_"+obj_task.replace(' ','')+'.csv')
    current_df.set_index("Region", inplace=True)
    current_df
    
    if groundtruths.index is None:
        groundtruths.set_index(current_df.index, inplace=True)
    
    groundtruths[obj_task] = current_df[obj_task]
    
#groundtruths_norm = (groundtruths - groundtruths.mean(axis=0)) / groundtruths.std(axis=0)
groundtruths.to_csv("groundtruths.csv")
#groundtruths_norm.to_csv("groundtruths_norm.csv")
display(groundtruths)
#display(groundtruths_norm)

In [ ]:
# Compute the mean and standard deviation over the rows of the original DataFrames
un_row_means = {}
un_row_std = {}
for column, df in column_dataframes.items():
    un_row_means[column] = df.mean(axis=1)
    un_row_std[column] = df.std(axis=1)
    
un_df = pd.DataFrame(un_row_means)

subj_df = un_df[tasks_subj_en]
obj_df = un_df[tasks_obj_en]
obj_df.to_csv(respath+"/obj_"+model+'.csv')
objrel_df = un_df[tasks_objrel_en]



In [ ]:
# Normalization of results


# Initialize a new dictionary to hold the normalized DataFrames
normalized_column_dataframes = {}

# Normalize each column by its mean
for column, df in column_dataframes.items():
    mean = df.mean()
    std = df.std()
    std.replace(0.0, 1.0, inplace=True)
    normalized_column_dataframes[column] = (df - mean)/ std

row_means = {}
row_std = {}
for column, df in normalized_column_dataframes.items():
    row_means[column] = df.mean(axis=1)
    row_std[column] = df.std(axis=1)
df_norm = pd.DataFrame(row_means)
#df_norm=df_norm.div(pd.DataFrame(row_std))

subj_norm_df = df_norm[tasks_subj_en]
subj_norm_df.to_csv(processed_resdir+"/subj_"+model+'.csv')
obj_norm_df = df_norm[tasks_obj_en]
objrel_norm_df = df_norm[tasks_objrel_en]
objrel_norm_df.to_csv(processed_resdir+"/objrel_"+model+'.csv')

display(subj_norm_df)

#rel_names = [t+' (Rating)' for t in tasks_obj_en]
#gt_rel = groundtruths_norm.copy()
#gt_rel.columns=rel_names
#diffs_rel = objrel_norm_df.sub(gt_rel)
#display(groundtruths_norm)

In [ ]:
#plotting groundtruths


cmap_ctr=0
for task in tasks_obj_en:
    plot_on_ger(groundtruths[task],task, task, plot_dir+"/"+model+'_'+task+"_gt.png", 1, cmap_abs[cmap_ctr%3])
    cmap_ctr+=1


In [ ]:
# Plotting results of objective tasks, absolute values

cmap_ctr=0
for task in tasks_obj_en:
    plot_on_ger(obj_df[task],task, task, plot_dir+"/"+model+'_'+task.replace(' ','')+".png", 1, cmap_abs[cmap_ctr%3])
    cmap_ctr+=1

In [ ]:
# Plotting results of objective tasks, ratings

cmap_ctr=0
for task in tasks_objrel_en:
    plot_on_ger(objrel_norm_df[task],task, task, plot_dir+"/"+model+'_'+task.replace(' ','')+".png", 1, cmap_rel[cmap_ctr%3])

    cmap_ctr+=1

In [ ]:
# Plotting results of subjective results

for task in tasks_subj_en:
    plot_on_ger(subj_norm_df[task],task, task, plot_dir+"/"+model+'_'+task+".png", 1, cmap_rel[cmap_ctr%3])
    cmap_ctr+=1

In [ ]:
# Plotting individual results of each run of one property; used e.g. to spot divergent behaviors in GPT4

task='Xenophobia'
for run in [model+str(n) for n in range(1,6)]:
    plot_on_ger(normalized_column_dataframes[task][run],run, task, plot_dir+"/"+run+'_'+task+".png", 1, cmap_rel[cmap_ctr%3])
    

In [ ]:
# test statistical significance

from scipy.stats import ttest_ind

# Results dictionary to store p-values and t-statistics for each metric
results = {}

# Loop through each metric
for metric, df in column_dataframes.items():
    # Segregate data for east states and other states
    east_data = df[df.index.isin(east_states)]
    other_data = df[~df.index.isin(east_states)]
    
    # Collect t-test results for each iteration within the metric
    metric_results = {}
    for column in df.columns:
        t_stat, p_value = ttest_ind(east_data[column], other_data[column], equal_var=False)
        metric_results[column] = (t_stat, p_value)
    
    # Store results for current metric
    results[metric] = metric_results

# Print results for review
for metric, data in results.items():
    print(f"Results for {metric}:")
    for run, res in data.items():
        print(f"  Run {run}: T-stat = {res[0]:.2f}, P-value = {res[1]:.4f}")
